### Modell anzeigen

In [1]:
from core.config import LLM_MODEL, OLLAMA_URL
print("LLM_MODEL =", LLM_MODEL)
print("OLLAMA_URL =", OLLAMA_URL)

LLM_MODEL = gpt-oss:120b-cloud
OLLAMA_URL = http://localhost:11434


### Anzeigen Welche Dokumente in ChromaDB `source` geben:

In [5]:
from pathlib import Path
from langchain_chroma import Chroma
from chromadb.config import Settings

from utils.ollama_embed import OllamaEmbeddings
from core.config import PATH_DB, COLLECTION, EMBED_MODEL, OLLAMA_URL, PATH_PROCESSED

# ------------------------------------------------------------------
# Create embedding model
# ------------------------------------------------------------------
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_URL
)

# ------------------------------------------------------------------
# Connect to vector database
# ------------------------------------------------------------------
vector_db = Chroma(
    collection_name=COLLECTION,
    persist_directory=PATH_DB,
    embedding_function=embeddings,
    client_settings=Settings(anonymized_telemetry=False),
)

# --- Nur Dokument-IDs anzeigen ---
res = vector_db.get(include=["metadatas"])
doc_ids = [m.get("source") for m in res["metadatas"] if m.get("source")]

# Dedupe, Reihenfolge behalten
unique_ids = list(dict.fromkeys(doc_ids))

processed_md_count = len(list(Path(PATH_PROCESSED).glob("*.md")))

print(f"{processed_md_count} Markdown-Dateien in {PATH_PROCESSED}")
print(f"{len(unique_ids)} insgesamt in DB-Collection:\n")

for doc in unique_ids:
    print(f"- {doc}")


3 Markdown-Dateien in data\processed
1 insgesamt in DB-Collection:

- dnk-datei-2024-124


### Aus ChromaDB ein bestimmtes Dokument als MD-Datei ausgeben:

In [1]:
from pathlib import Path
from langchain_chroma import Chroma
from chromadb.config import Settings

from utils.ollama_embed import OllamaEmbeddings
from core.config import PATH_DB, COLLECTION, EMBED_MODEL, OLLAMA_URL

# ------------------------------------------------------------------
# Create embedding model
# ------------------------------------------------------------------
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_URL
)

# ------------------------------------------------------------------
# Connect to vector database
# ------------------------------------------------------------------
vector_db = Chroma(
    collection_name=COLLECTION,
    persist_directory=PATH_DB,
    embedding_function=embeddings,
    client_settings=Settings(anonymized_telemetry=False),
)

# ------------------------------------------------------------------
# Ziel-Dokument
# ------------------------------------------------------------------
target_id = "dnk-datei-2024-124" #                  <----------- hier eingeben!
print("Gesuchtes Dokument:", target_id)

# ------------------------------------------------------------------
# Daten laden (bewusst nur documents + metadatas)
# ------------------------------------------------------------------
res = vector_db.get(include=["documents", "metadatas"])

# ------------------------------------------------------------------
# Filtern nach source
# ------------------------------------------------------------------
chunks = [
    doc
    for doc, meta in zip(res["documents"], res["metadatas"])
    if meta.get("source") == target_id
]

print(f"{len(chunks)} Chunks gefunden")

# ------------------------------------------------------------------
# Markdown-Datei schreiben
# ------------------------------------------------------------------
out_path = Path(f"{target_id}.md")

with out_path.open("w", encoding="utf-8") as f:
    # Dokument-Kopf
    f.write(f"# Dokument: {target_id}\n")
    f.write(f"- Anzahl Chunks: {len(chunks)}\n\n")

    # Chunks
    for i, text in enumerate(chunks, 1):
        f.write(f"{'=' * 30} {i} {'=' * 30}\n\n")
        f.write(text.strip())
        f.write("\n\n")

print(f"Markdown-Datei geschrieben: {out_path.resolve()}")


Gesuchtes Dokument: dnk-datei-2024-124
0 Chunks gefunden
Markdown-Datei geschrieben: C:\Users\Mohammad Taiba\PycharmProjects\rag2-app\dnk-datei-2024-124.md


### Aus ChromaDB ein bestimmtes Dokument löschen:

In [3]:
target_source = "dnk-2024-wildeboer-bauteile-gmbh"  #                          <------------ hier eingeben

# 1) Alle IDs + Metadaten holen
res = vector_db.get(include=["metadatas"])  # KEIN include=["ids"] !

ids_to_delete = [
    _id
    for _id, meta in zip(res["ids"], res["metadatas"])
    if meta.get("source") == target_source
]

print(f"Treffer zum Löschen für source='{target_source}': {len(ids_to_delete)}")

# 2) Löschen (nur wenn Treffer)
if ids_to_delete:
    vector_db.delete(ids=ids_to_delete)

    # je nach Setup nötig/harmlos
    try:
        vector_db.persist()
    except Exception:
        pass

    print("Gelöscht.")
else:
    print("Nichts zu löschen.")


Treffer zum Löschen für source='dnk-2024-wildeboer-bauteile-gmbh': 1266
Gelöscht.


### Test Chunking

In [2]:
# utils/chunking.py
from langchain_text_splitters import MarkdownHeaderTextSplitter
from utils.logger import logger

def chunk_documents(docs):
    logger.info("Chunking beginnt ...")

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#", "H1"), ("##", "H2"), ("###", "H3")],
        strip_headers=False
    )

    chunks = [] # An empty list is created in which all generated chunks are collected.
    for doc in docs:
        for chunk in splitter.split_text(doc.page_content):
            chunk.metadata["source"] = doc.metadata.get("source") # Add  metadata `source` for example: `dokument_1.md`
            chunks.append(chunk) # Save the new chunk

    logger.info("Chunking abgeschlossen.")
    return chunks


In [3]:
markdown_document = """
    # Das ist Header 1 üüü
    text_Header1

    ## Das ist Header 2
    text_Header2

    ### Das ist Header 3
    text_Header3
"""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits

[Document(metadata={'Header 1': 'Das ist Header 1 üüü'}, page_content='# Das ist Header 1 üüü\ntext_Header1'),
 Document(metadata={'Header 1': 'Das ist Header 1 üüü', 'Header 2': 'Das ist Header 2'}, page_content='## Das ist Header 2\ntext_Header2'),
 Document(metadata={'Header 1': 'Das ist Header 1 üüü', 'Header 2': 'Das ist Header 2', 'Header 3': 'Das ist Header 3'}, page_content='### Das ist Header 3\ntext_Header3')]

### Chunks-Länge überprüfen

In [2]:
from chromadb import PersistentClient
from core.config import PATH_DB, COLLECTION

client = PersistentClient(path=PATH_DB)
collection = client.get_collection(COLLECTION)

items = collection.get(include=["documents"])
docs = items["documents"]

# grober Überblick
lengths = [len(d) for d in docs]
print("Min:", min(lengths))
print("Max:", max(lengths))
print("Avg:", sum(lengths) / len(lengths))

# Anzahl Chunks anzeigen
short_chunks_length = 300
short_chunks = [d for d in docs if len(d) < short_chunks_length]
print(f"\nAnzahl gesamte Chunks: {len(docs)}")
print(f"Anzahl kurzer  Chunks, die kleine als {short_chunks_length}: {len(short_chunks)}")

# Ausgabe der kurzen Chunks
i = 1
for d in docs:
    if len(d) < short_chunks_length:
        print(f"{i} ---------------------------------------\n", d)
        i = i + 1

Min: 37
Max: 2032
Avg: 1419.1464968152866

Anzahl gesamte Chunks: 157
Anzahl kurzer  Chunks, die kleine als 300: 5
1 ---------------------------------------
 ### 3. Ziele de  
Das Unternehmen legt offen, welche qualitativen und/oder quantitativen sowie zeitlich definierten Nachhaltigkeitsziele gesetzt und operationalisiert werden und wie deren Erreichungsgrad kontrolliert wird.

---
Quelle: dnk_datei_2024_1.md

2 ---------------------------------------
 ###

---
Quelle: dnk_datei_2024_1.md

3 ---------------------------------------
 ### Überwachung und Kontrolle der Zielerreichung

---
Quelle: dnk_datei_2024_1.md

4 ---------------------------------------
 ### 4. Tiefe der Wertschöpfungskette de  
Das Unternehmen gibt an, welche Bedeutung Aspekte der Nachhaltigkeit für die Wertschöpfung haben und bis zu welcher Tiefe seiner Wertschöpfungskette Nachhaltigkeitskriterien überprüft werden.

---
Quelle: dnk_datei_2024_1.md

5 ---------------------------------------
 Als Teil der global agie

### Hash password: (SHA256)

In [6]:
import hashlib
print(hashlib.sha256("DEIN_PASSWORT".encode()).hexdigest())

6cb260d31b7f84e4f0fbd94661b9be2143e051b77ebbcc15e6b294e545704f10


### ChromaDB komplett als eine MD-Datei ausgeben:

In [3]:
from __future__ import annotations

from contextlib import redirect_stdout
from pathlib import Path

from langchain_chroma import Chroma
from core.config import COLLECTION, PATH_DB

# Log-Verzeichnis sicherstellen
Path("logs").mkdir(exist_ok=True)

# Logdatei
log_path = Path("logs/chroma_output.md")

# ChromaDB öffnen
db = Chroma(
    collection_name=COLLECTION,
    persist_directory=PATH_DB,
)

print("Collection:", COLLECTION)
print("DB-Pfad:", PATH_DB)
print("Anzahl Chunks:", db._collection.count())


# Ausgabe in Markdown-Datei schreiben
with open(log_path, "w", encoding="utf-8") as f, redirect_stdout(f):
    print("ChromaDB Übersicht\n")
    print(f"- **Gesamtzahl:** {db._collection.count()}")
    print("\nChunks-Embeddings\n")

    res = db._collection.get(include=["documents", "metadatas"])

    for i, doc in enumerate(res.get("documents", []) or []):
        print(f"> Chunk {i + 1}:\n")
        print(doc)
        print("\n")

print(f"✅ Ausgabe gespeichert unter: {log_path.resolve()}")

Collection: bachelorarbeit
DB-Pfad: db\chromadb
Anzahl Chunks: 157
✅ Ausgabe gespeichert unter: C:\Users\moham\PycharmProjects\rag2-app\logs\chroma_output.md
